# Session 3: designing the experiment, and checking it by simulation
**A. Real-world problem.** A sensor drifts slowly while you work. If you test all of
treatment A first and all of treatment B afterwards, the drift looks like a treatment
effect. Good design (randomise, balance, replicate, vary factors together) protects you
before any data exist. We test each design on a simulated world where the true answer is
known, so we can measure bias and scatter of the answer each design gives.

**Plan.** (1) run order under drift, (2) how many replicates, (3) a 2x2 factorial design
versus one-factor-at-a-time, (4) calibration drift and re-calibration, (5) a failure case,
(6) a two-factor calibration (the mini-project).

In [ ]:
import math
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# --- part 1: run order under drift ----------------------------------------------------------
N_RUNS = 20          # 10 runs of treatment A (code 0) and 10 of B (code 1)
TRUE_EFFECT = 0.30   # B reads higher than A by this much (degrees C)
DRIFT = 0.05         # sensor drift per run (degrees C per run)
NOISE = 0.50         # random scatter of one run (degrees C)

## Step 1: five run orders
`naive` = all A then all B. `alternating` = ABAB... `abba` = ABBA ABBA... (a drift that is
a straight line cancels exactly). `random` = shuffle all 20. `blocked` = in every pair of
neighbouring runs one A and one B, in random order (randomised blocks).

In [ ]:
def make_order(kind, rng):
    """Array of 0 (A) and 1 (B) of length N_RUNS."""
    half = N_RUNS // 2
    if kind == "naive":
        return np.array([0] * half + [1] * half)
    if kind == "alternating":
        return np.array([0, 1] * half)
    if kind == "abba":
        return np.array([0, 1, 1, 0] * (N_RUNS // 4))
    if kind == "random":
        return rng.permutation(np.array([0] * half + [1] * half))
    if kind == "blocked":
        pairs = [rng.permutation([0, 1]) for _ in range(half)]
        return np.concatenate(pairs)
    raise ValueError(kind)


def effect_estimate(order, rng, drift=DRIFT, effect=TRUE_EFFECT, noise=NOISE):
    """Run the experiment once: mean(B) - mean(A) in the drifting world."""
    t = np.arange(N_RUNS)
    y = effect * order + drift * t + rng.normal(0.0, noise, N_RUNS)
    return float(y[order == 1].mean() - y[order == 0].mean())


def order_study(kind, reps, seed, drift=DRIFT):
    """(bias, scatter) of the effect estimate over many repeats of the whole experiment."""
    rng = np.random.default_rng(seed)
    est = np.array([effect_estimate(make_order(kind, rng), rng, drift=drift) for _ in range(reps)])
    return float(est.mean() - TRUE_EFFECT), float(est.std(ddof=1))


def drift_bias(order, drift=DRIFT):
    """Exact bias a straight-line drift adds to mean(B) - mean(A): drift * (mean time of B - mean time of A)."""
    t = np.arange(len(order))
    return float(drift * (t[order == 1].mean() - t[order == 0].mean()))

## Step 2: replication and power
To detect a true difference d between two group means (each run has scatter sigma), with
n runs per group, a two-sided z-test at 5 percent significance with power 80 percent needs

    n = 2 (z_a + z_b)^2 sigma^2 / d^2,   z_a = 1.96 (5 percent two-sided), z_b = 0.8416 (80 percent power).

*Power* is the chance the test finds the difference when it is real. We check the formula
with a seeded Monte Carlo simulation (known sigma, so a z-test).

In [ ]:
ZA = 1.959964
ZB = 0.841621


def sample_size(d, sigma, za=ZA, zb=ZB):
    return int(math.ceil(2.0 * (za + zb) ** 2 * sigma ** 2 / d ** 2 - 1e-12))


def phi(z):
    """Standard normal cumulative distribution."""
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def power_formula(n, d, sigma, za=ZA):
    shift = d / (sigma * math.sqrt(2.0 / n))
    return phi(shift - za) + phi(-shift - za)


def simulate_power(n, d, sigma, reps, seed, za=ZA):
    rng = np.random.default_rng(seed)
    a = rng.normal(0.0, sigma, (reps, n)).mean(axis=1)
    b = rng.normal(d, sigma, (reps, n)).mean(axis=1)
    z = (b - a) / (sigma * math.sqrt(2.0 / n))
    return float(np.mean(np.abs(z) > za))

## Step 3: 2x2 factorial design versus one factor at a time
Two factors A and B, each at a low (-1) and high (+1) setting, give four runs:
(-,-), (+,-), (-,+), (+,+). The main effect of A is mean(response at A high) minus
mean(response at A low); the interaction AB is half the difference between the effect of
A when B is high and the effect of A when B is low. The made-up system below has
response = 50 + A + B + 6 A B: the two factors only help when changed together.

In [ ]:
def system(a, b):
    """Noise-free response of the made-up system (a, b are -1 or +1)."""
    return 50.0 + a + b + 6.0 * a * b


RUNS = [(-1, -1), (1, -1), (-1, 1), (1, 1)]


def factorial_effects(y):
    """y = responses of the four runs in the order of RUNS. Returns (A, B, AB) effects."""
    y1, y2, y3, y4 = y
    ea = ((y2 + y4) - (y1 + y3)) / 2.0
    eb = ((y3 + y4) - (y1 + y2)) / 2.0
    eab = ((y1 + y4) - (y2 + y3)) / 2.0
    return ea, eb, eab


def one_factor_at_a_time(f=system):
    """Start at (-1,-1); change A and keep the better; then change B and keep the better.
    Returns (a, b, response) of the setting it ends at."""
    a, b = -1, -1
    if f(1, b) > f(a, b):
        a = 1
    if f(a, 1) > f(a, b):
        b = 1
    return a, b, f(a, b)

## Step 4: calibration drift and the re-calibration schedule
Right after calibration the offset error is random with sd U_CAL; then it drifts at RATE
degrees per day. We want the error to stay within TOL. Using 1.96 * U_CAL as the worst
starting error, we must re-calibrate before 1.96 * U_CAL + RATE * t reaches TOL.

In [ ]:
U_CAL = 0.10   # standard uncertainty of the offset right after calibration (degrees C)
RATE = 0.02    # drift (degrees C per day)
TOL = 0.50     # allowed error (degrees C)


def recal_days(tol=TOL, u_cal=U_CAL, rate=RATE, z=ZA):
    """Largest whole number of days t with z * u_cal + rate * t <= tol."""
    return max(0, int(math.floor((tol - z * u_cal) / rate + 1e-9)))


def p_exceed(t, tol=TOL, u_cal=U_CAL, rate=RATE):
    """Chance the error is above +tol after t days (upper tail only)."""
    return 1.0 - phi((tol - rate * t) / u_cal)


def simulate_exceed(t, reps, seed, tol=TOL, u_cal=U_CAL, rate=RATE):
    rng = np.random.default_rng(seed)
    return float(np.mean(rng.normal(0.0, u_cal, reps) + rate * t > tol))

## Step 5: a failure case - extrapolating a nonlinear sensor
The made-up sensor below bends: reading = T + 0.002 T^2 (degrees C). Calibrated with a
straight line on 0..50 degrees it looks fine inside that range, but outside it the line
is badly wrong. The residuals show a systematic curve - the warning sign.

In [ ]:
def curved(T):
    T = np.asarray(T, dtype=float)
    return T + 0.002 * T ** 2


def extrapolation_demo(lo=0.0, hi=50.0, n=11):
    """Fit a line on [lo, hi]; returns (slope, intercept, max |residual| inside, error at 100)."""
    x = np.linspace(lo, hi, n)
    y = curved(x)
    m, c = np.polyfit(x, y, 1)
    inside = float(np.max(np.abs(y - (m * x + c))))
    err100 = float(curved(100.0) - (m * 100.0 + c))
    return float(m), float(c), inside, err100

## Step 6: mini-project - calibrate across temperature and supply voltage
The made-up sensor also depends on its supply voltage V (volts) around 5 V:

    reading = 0.98 T + 0.5 + (V - 5) (0.8 + 0.004 T) + drift + noise

We run a 2x2 factorial: T in {20, 80} degrees C and V in {4.5, 5.5} V, 5 replicates per
corner (20 runs), in randomised blocks (every block of 4 runs has each corner once). We
estimate the effects, then fit reading = b0 + b1 T + b2 dV + b3 T dV (dV = V - 5) by
least squares and invert it for T when V is known.

In [ ]:
TS = (20.0, 80.0)
VS = (4.5, 5.5)
SENSOR_NOISE = 0.30
RUN_DRIFT = 0.01   # sensor drift per run (degrees C per run)


def reading2(T, V, run=0, rng=None, noise=SENSOR_NOISE):
    r = 0.98 * T + 0.5 + (V - 5.0) * (0.8 + 0.004 * T) + RUN_DRIFT * run
    if rng is not None:
        r = r + rng.normal(0.0, noise, np.shape(r))
    return r


def design_runs(reps, rng):
    """Randomised blocks: list of (T, V) for 4*reps runs."""
    corners = [(t, v) for t in TS for v in VS]
    runs = []
    for _ in range(reps):
        for k in rng.permutation(4):
            runs.append(corners[k])
    return runs


def run_experiment(reps, seed):
    rng = np.random.default_rng(seed)
    runs = design_runs(reps, rng)
    T = np.array([r[0] for r in runs])
    V = np.array([r[1] for r in runs])
    y = reading2(T, V, np.arange(len(runs)), rng)
    return T, V, y


def two_factor_effects(T, V, y):
    """Main effects of T and V and their interaction (standard 2x2 definitions), from corner means."""
    def mean_at(t, v):
        return float(y[(T == t) & (V == v)].mean())
    y1, y2, y3, y4 = (mean_at(TS[0], VS[0]), mean_at(TS[1], VS[0]), mean_at(TS[0], VS[1]), mean_at(TS[1], VS[1]))
    return factorial_effects((y1, y2, y3, y4))


def fit_two_factor(T, V, y):
    """Least squares for [b0, b1, b2, b3] in reading = b0 + b1 T + b2 dV + b3 T dV; and residual sd."""
    dV = V - 5.0
    X = np.column_stack([np.ones_like(T), T, dV, T * dV])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ b
    s = math.sqrt(float(resid @ resid) / (len(y) - 4))
    return b, s


def estimate_T(b, reading, V):
    dV = V - 5.0
    return (reading - b[0] - b[2] * dV) / (b[1] + b[3] * dV)


def mini_project_stats(reps, trials, seed, T_test=50.0, V_test=5.4):
    """Over many repeats of the whole experiment: (sd of T error with V-aware calibration,
    bias of T if V is ignored). A single new reading at (T_test, V_test) is inverted."""
    rng = np.random.default_rng(seed)
    errs = []
    for _ in range(trials):
        T, V, y = run_experiment(reps, int(rng.integers(1 << 30)))
        b, _s = fit_two_factor(T, V, y)
        ynew = float(reading2(T_test, V_test, 25, rng))
        errs.append(estimate_T(b, ynew, V_test) - T_test)
    errs = np.array(errs)
    # calibration that ignores V: line through all runs, evaluated for the same reading
    T, V, y = run_experiment(reps, seed)
    m, c = np.polyfit(T, y, 1)
    naive_err = (float(reading2(T_test, V_test, 25)) - c) / m - T_test
    return float(errs.std(ddof=1)), float(naive_err)

## Step 7: see it
Four figures: run orders under drift, power against n, the 2x2 interaction plot, and the
extrapolation failure.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    kinds = ["naive", "alternating", "abba", "random", "blocked"]
    bias, sd = [], []
    for k in kinds:
        b_, s_ = order_study(k, 4000, 3)
        bias.append(b_)
        sd.append(s_)
        print("%-12s bias %+.3f  scatter %.3f" % (k, b_, s_))
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
    axs[0].bar(kinds, bias)
    axs[0].axhline(0, color="k", lw=0.8)
    axs[0].set_ylabel("bias of effect estimate (degrees C)")
    axs[0].set_title("Bias caused by drift")
    axs[1].bar(kinds, sd, color="tab:orange")
    axs[1].set_ylabel("scatter of effect estimate (degrees C)")
    axs[1].set_title("Scatter of the estimate")
    for a in axs:
        a.tick_params(axis="x", labelrotation=30)
        a.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "run_orders.png", dpi=120)

    print("sample size d=0.5, sigma=1:", sample_size(0.5, 1.0),
          "power", round(power_formula(63, 0.5, 1.0), 3), "sim", round(simulate_power(63, 0.5, 1.0, 20000, 1), 3))
    ns = np.arange(2, 120)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ns, [power_formula(int(n), 0.5, 1.0) for n in ns], label="formula")
    ax2.plot(ns[::8], [simulate_power(int(n), 0.5, 1.0, 5000, 2) for n in ns[::8]], "o", label="simulation")
    ax2.axhline(0.8, color="gray", ls=":")
    ax2.axvline(63, color="gray", ls=":")
    ax2.set_xlabel("runs per group, n")
    ax2.set_ylabel("power (chance to detect d = 0.5, sigma = 1)")
    ax2.set_title("Power against replication")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "power_vs_n.png", dpi=120)

    fig3, ax3 = plt.subplots(figsize=(5.5, 4))
    for bv, st in ((-1, "--"), (1, "-")):
        ax3.plot([-1, 1], [system(-1, bv), system(1, bv)], st, marker="o", label="B = %+d" % bv)
    ax3.set_xlabel("factor A (coded level)")
    ax3.set_ylabel("response")
    ax3.set_title("Interaction: lines are not parallel")
    ax3.legend()
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "interaction.png", dpi=120)
    print("factorial effects:", factorial_effects([system(a, b) for a, b in RUNS]), "OFAT ends at", one_factor_at_a_time())

    m, c, inside, err100 = extrapolation_demo()
    xs = np.linspace(0, 100, 101)
    fig4, ax4 = plt.subplots(figsize=(6, 4))
    ax4.plot(xs, curved(xs), label="real (curved) sensor")
    ax4.plot(xs, m * xs + c, "--", label="line fitted on 0..50")
    ax4.axvspan(0, 50, color="green", alpha=0.1, label="calibrated range")
    ax4.set_xlabel("true temperature (degrees C)")
    ax4.set_ylabel("sensor reading (degrees C)")
    ax4.set_title("Extrapolation error at 100 degrees C: %.1f" % err100)
    ax4.legend(fontsize=8)
    ax4.grid(alpha=0.3)
    fig4.tight_layout()
    fig4.savefig(OUT / "extrapolation.png", dpi=120)
    print("recal every", recal_days(), "days; P(exceed) at 15 days:", round(p_exceed(15), 4))
    T_, V_, y_ = run_experiment(5, 1)
    print("effects (T, V, TV):", np.round(two_factor_effects(T_, V_, y_), 3))
    print("calibration fit b, s:", fit_two_factor(T_, V_, y_))
    print("mini-project:", mini_project_stats(5, 400, 7))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `order_study(kind, 4000, 3)` for the five orders with `drift` 0 and compare with drift 0.05. Which orders change?
2. **Stretch.** Use `sample_size` and `power_formula` to find how many runs per group are needed for d = 0.5 when sigma is 2.0, and check with `simulate_power`.
3. **Explain.** Explain, with the 2x2 numbers 54, 44, 44, 58, why the main effect of A is small (2) although changing A alone at the start of the design makes the response fall by 10.

## Assessment
- **A. Conceptual.** Why does randomising the run order protect an experiment against drift you did not know about, while ABBA only protects against straight-line drift? What does randomisation cost?
- **B. Numerical.** How many runs per group are needed to detect a difference d = 0.25 with sigma = 1.0 at 5 percent significance and 80 percent power? Compare with d = 0.5 (63 runs).
- **C. Graph and simulation interpretation.** In the interaction figure the two lines are not parallel. What does this say about the effect of factor A, and why does one-factor-at-a-time fail here?
- **D. Design and optimisation challenge.** Design: after calibration the offset uncertainty is 0.10 degrees C, the drift is 0.02 degrees C per day and the allowed error is 0.50 degrees C. How often must you recalibrate? What happens after 20 days? What if the tolerance were 0.10?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    rng = np.random.default_rng(0)

    # --- run orders ---
    for kind in ("naive", "alternating", "abba", "random", "blocked"):
        o = L.make_order(kind, rng)
        if len(o) != 20 or o.sum() != 10:
            checks.append(("every run order has 20 runs, 10 of each treatment (%s)" % kind, False))
            break
    else:
        checks.append(("every run order has 20 runs, 10 of each treatment", True))
    checks.append(("worked example: all-A-then-all-B adds a drift bias of 0.05 * 10 = 0.50 degrees C",
                   abs(L.drift_bias(L.make_order("naive", rng)) - 0.50) < 1e-12))
    checks.append(("alternating ABAB adds 0.05 and ABBA adds exactly 0",
                   abs(L.drift_bias(L.make_order("alternating", rng)) - 0.05) < 1e-12
                   and abs(L.drift_bias(L.make_order("abba", rng))) < 1e-12))
    res = {k: L.order_study(k, 4000, 3) for k in ("naive", "alternating", "abba", "random", "blocked")}
    checks.append(("simulation: naive order estimates the effect 0.50 too high (bias 0.50 +- 0.02)",
                   abs(res["naive"][0] - 0.50) < 0.02))
    checks.append(("simulation: ABBA, random and blocked orders have bias below 0.02 in size",
                   all(abs(res[k][0]) < 0.02 for k in ("abba", "random", "blocked"))))
    checks.append(("simulation: alternating order has a small bias, about 0.05",
                   abs(res["alternating"][0] - 0.05) < 0.02))
    noise_only = 0.5 * math.sqrt(2.0 / 10.0)
    checks.append(("hand calculation: noise alone gives scatter 0.5 * sqrt(2/10) = 0.2236",
                   abs(noise_only - 0.2236) < 0.0001 and abs(res["abba"][1] / noise_only - 1) < 0.05))
    rand_pred = math.sqrt(noise_only ** 2 + 0.05 ** 2 * 7.0)
    checks.append(("hand calculation: full randomisation adds drift scatter 0.05 * sqrt(7): total 0.260",
                   abs(rand_pred - 0.2598) < 0.0005 and abs(res["random"][1] / rand_pred - 1) < 0.05))
    checks.append(("randomised blocks keep the scatter near the noise-only value (below 0.235) and below full randomisation",
                   res["blocked"][1] < 0.235 and res["blocked"][1] < res["random"][1]))
    z = np.mean([L.effect_estimate(L.make_order("naive", rng), rng, drift=0.0) for _ in range(4000)])
    checks.append(("without drift, even the naive order finds the true effect 0.30 (mean 0.29 to 0.31)",
                   abs(z - 0.30) < 0.015))

    # --- power ---
    checks.append(("worked example: d = 0.5, sigma = 1, 5 percent, 80 percent power needs n = 63 per group",
                   L.sample_size(0.5, 1.0) == 63))
    checks.append(("hand calculation: 2 * (1.96 + 0.8416)^2 / 0.25 = 62.79, rounded up to 63",
                   abs(2 * (1.96 + 0.8416) ** 2 / 0.25 - 62.79) < 0.02))
    checks.append(("halving d quadruples n: d = 0.25 needs 252", L.sample_size(0.25, 1.0) == 252))
    checks.append(("n = 63 has power 0.801, n = 62 has 0.795 (formula)",
                   abs(L.power_formula(63, 0.5, 1.0) - 0.8013) < 0.0005 and abs(L.power_formula(62, 0.5, 1.0) - 0.7950) < 0.0005))
    checks.append(("seeded Monte Carlo power at n = 63 is 0.80 (within 0.015 of the formula)",
                   abs(L.simulate_power(63, 0.5, 1.0, 20000, 1) - L.power_formula(63, 0.5, 1.0)) < 0.015))
    checks.append(("only 16 runs per group give power 0.29, so most real differences would be missed",
                   abs(L.power_formula(16, 0.5, 1.0) - 0.293) < 0.001 and abs(L.simulate_power(16, 0.5, 1.0, 20000, 1) - 0.293) < 0.015))
    checks.append(("edge case: with no true difference the test fires 5 percent of the time (0.05 +- 0.01)",
                   abs(L.simulate_power(63, 0.0, 1.0, 20000, 1) - 0.05) < 0.01))

    # --- factorial ---
    y = [L.system(a, b) for a, b in L.RUNS]
    checks.append(("worked example: the four runs give 54, 44, 44, 58", y == [54.0, 44.0, 44.0, 58.0]))
    checks.append(("effects from the four runs: A = 2, B = 2, AB = 12", L.factorial_effects(y) == (2.0, 2.0, 12.0)))
    checks.append(("one-factor-at-a-time stops at (-1, -1) with response 54", L.one_factor_at_a_time() == (-1, -1, 54.0)))
    checks.append(("the best setting is (+1, +1) with response 58, which one-factor-at-a-time misses",
                   max(y) == L.system(1, 1) == 58.0))
    checks.append(("without an interaction one-factor-at-a-time finds the best setting",
                   L.one_factor_at_a_time(lambda a, b: 50.0 + a + b) == (1, 1, 52.0)))
    r2 = np.random.default_rng(2)
    e = np.array([L.factorial_effects([L.system(a, b) + r2.normal(0, 1) for a, b in L.RUNS]) for _ in range(20000)])
    checks.append(("simulated: with noise sd 1 the effect estimates are unbiased and each has sd about 1",
                   np.allclose(e.mean(0), [2, 2, 12], atol=0.03) and np.allclose(e.std(0), 1.0, atol=0.03)))

    # --- drift and re-calibration ---
    checks.append(("worked example: re-calibrate every 15 days (1.96 * 0.10 + 0.02 * t <= 0.50)", L.recal_days() == 15))
    checks.append(("hand calculation: (0.50 - 0.196) / 0.02 = 15.2 days", abs((0.50 - 0.196) / 0.02 - 15.2) < 0.001))
    checks.append(("after 15 days the chance of being above +0.50 is 0.0228",
                   abs(L.p_exceed(15) - 0.0228) < 0.0001 and abs(L.simulate_exceed(15, 400000, 3) - 0.0228) < 0.002))
    checks.append(("after 20 days that chance has grown to 0.159", abs(L.p_exceed(20) - 0.1587) < 0.0005))
    checks.append(("edge case: an impossible tolerance (0.10) gives 0 days", L.recal_days(0.10, 0.10, 0.02) == 0))

    # --- extrapolation ---
    m, c, inside, err100 = L.extrapolation_demo()
    checks.append(("line fitted on 0..50 of the curved sensor: slope 1.10, intercept -0.75",
                   abs(m - 1.10) < 1e-9 and abs(c + 0.75) < 1e-9))
    checks.append(("inside 0..50 the largest residual is only 0.75 degrees C", abs(inside - 0.75) < 1e-9))
    checks.append(("at 100 degrees C the extrapolated line is wrong by 10.75 degrees C", abs(err100 - 10.75) < 1e-9))
    checks.append(("a straight sensor has no extrapolation problem (curvature 0 gives error 0)",
                   abs(np.polyfit([0, 25, 50], [0, 25, 50], 1)[0] * 100 - 100) < 1e-9))

    # --- mini-project ---
    T, V, yy = L.run_experiment(5, 1)
    checks.append(("mini-project design: 20 runs, each corner 5 times",
                   len(T) == 20 and all(np.sum((T == t) & (V == v)) == 5 for t in L.TS for v in L.VS)))
    eT, eV, eTV = L.two_factor_effects(T, V, yy)
    se = L.SENSOR_NOISE / math.sqrt(5)
    checks.append(("worked example: effects near the true 58.8 (T), 1.0 (V), 0.12 (T x V), each within 3 standard errors (0.13)",
                   abs(eT - 58.8) < 3 * se and abs(eV - 1.0) < 3 * se and abs(eTV - 0.12) < 3 * se))
    checks.append(("hand calculation: true T effect = 0.98 * 60 = 58.8; V effect = 0.8 + 0.004 * 50 = 1.0",
                   abs(0.98 * 60 - 58.8) < 1e-9 and abs(0.8 + 0.004 * 50 - 1.0) < 1e-9))
    b, s = L.fit_two_factor(T, V, yy)
    checks.append(("fitted coefficients near the truth (b1 = 0.98 within 0.01, b2 = 0.80 within 0.3); residual sd near 0.30",
                   abs(b[1] - 0.98) < 0.01 and abs(b[2] - 0.80) < 0.3 and abs(s - 0.30) < 0.08))
    check_x = L.estimate_T(np.array([0.5, 0.98, 0.8, 0.004]), 0.98 * 50 + 0.5 + 0.4 * (0.8 + 0.2), 5.4)
    checks.append(("inverting the exact model returns the temperature (50)", abs(check_x - 50.0) < 1e-9))
    sd_aware, naive_err = L.mini_project_stats(5, 400, 7)
    checks.append(("mini-project: with the voltage in the model, the error of one reading at 50 C, 5.4 V has sd 0.29 degrees C",
                   abs(sd_aware - 0.29) < 0.03))
    checks.append(("mini-project: ignoring the voltage gives a temperature error of 0.74 degrees C at 5.4 V",
                   abs(naive_err - 0.74) < 0.02))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
